# # Day 1: Data Curation for Price Prediction

### This notebook performs the first curation step for Amazon product metadata:
### - load raw source data
### - parse and clean records into `Item` objects
### - build a weighted curated sample
### - split into train/validation/test
### - push the curated dataset to Hugging Face Hub

In [3]:
import os
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from huggingface_hub import login

from pricer.items import Item
from pricer.parser import parse

In [4]:
def authenticate_huggingface() -> None:
    """Authenticate to Hugging Face Hub using the HF_TOKEN environment variable."""
    hf_token = os.environ["HF_TOKEN"]
    login(hf_token, add_to_git_credential=True)


def load_raw_appliances_dataset():
    """Load the raw Amazon Appliances metadata split from Hugging Face datasets."""
    return load_dataset(
        "McAuley-Lab/Amazon-Reviews-2023",
        "raw_meta_Appliances",
        split="full",
        trust_remote_code=True,
    )


def parse_valid_items(dataset, category: str = "Appliances") -> list[Item]:
    """Parse dataset records into valid Item objects using project parsing rules."""
    parsed_items = [parse(datapoint, category) for datapoint in tqdm(dataset)]
    valid_items = [item for item in parsed_items if item is not None]
    print(f"Parsed {len(valid_items):,} valid items from {len(dataset):,} rows")
    return valid_items


def build_weighted_sample(items: list[Item], sample_size: int, seed: int = 42) -> list[Item]:
    """Create a weighted sample that favors higher-priced products with category balancing."""
    if sample_size > len(items):
        raise ValueError("sample_size cannot be greater than number of items")

    np.random.seed(seed)
    prices = np.array([item.price for item in items], dtype=float)
    categories = np.array([item.category for item in items])

    normalized_prices = (prices - prices.min()) / (prices.max() - prices.min() + 1e-9)
    weights = normalized_prices**2
    weights[categories == "Tools_and_Home_Improvement"] *= 0.5
    weights[categories == "Automotive"] *= 0.05
    weights = weights / weights.sum()

    selected_indices = np.random.choice(len(items), size=sample_size, replace=False, p=weights)
    return [items[index] for index in selected_indices]


def split_items(
    items: list[Item],
    train_ratio: float = 0.98,
    validation_ratio: float = 0.01,
    seed: int = 42,
) -> tuple[list[Item], list[Item], list[Item]]:
    """Shuffle and split items into train, validation, and test sets."""
    if not 0 < train_ratio < 1:
        raise ValueError("train_ratio must be between 0 and 1")
    if not 0 < validation_ratio < 1:
        raise ValueError("validation_ratio must be between 0 and 1")
    if train_ratio + validation_ratio >= 1:
        raise ValueError("train_ratio + validation_ratio must be less than 1")

    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(items))
    shuffled_items = [items[index] for index in indices]

    train_end = int(len(shuffled_items) * train_ratio)
    validation_end = train_end + int(len(shuffled_items) * validation_ratio)

    train_items = shuffled_items[:train_end]
    validation_items = shuffled_items[train_end:validation_end]
    test_items = shuffled_items[validation_end:]
    return train_items, validation_items, test_items


def push_curated_splits(
    dataset_name: str,
    train_items: list[Item],
    validation_items: list[Item],
    test_items: list[Item],
) -> None:
    """Push curated train, validation, and test splits to Hugging Face Hub."""
    Item.push_to_hub(dataset_name, train_items, validation_items, test_items)
    print(
        f"Pushed dataset to {dataset_name} with "
        f"train={len(train_items):,}, validation={len(validation_items):,}, test={len(test_items):,}"
    )

In [5]:
dataset = load_raw_appliances_dataset()

In [7]:
print(dataset[10000])

{'main_category': 'Industrial & Scientific', 'title': 'Caynel Countertop Ice Maker – 9 Ice Cubes Ready in 6 Mins - 26LBS/24Hrs, 2 Ice Sizes, LED Indicator Lights – Portable Ice Maker with Ice Scoop and Basket for Home Office Bar Party, Silver', 'average_rating': 3.9, 'rating_number': 26, 'features': ['DESIGNED TO MAKE PERFECT ICE FAST - Caynel Ice Maker works quickly making 9 chewable delicious bullet-shaped ice-cubes in one cycle in just 6 minutes—and up to 26 lbs of ice per day. Two selectable ice cube sizes making it perfect for mixed drinks or small water bottle openings.', 'INTELLIGENT FEATURES - Quickly respond to your every tap of corresponding button on operation area. Our ice maker prevents overflowing with warning lights and an automatic shut-off feature for when the ice basket is full or if water needs to be refilled, so as to reduce mistakes and protect device itself to prolong life span. Press and hold the power button 3 seconds to enter the automatic cleaning function.', 

In [ ]:
items = parse_valid_items(dataset)


In [23]:
len(items)

35307

In [ ]:
for item in items[:5]:
    print(item)

In [ ]:
curated_sample = build_weighted_sample(items, sample_size=10001, seed=42)

print(len(curated_sample))

for item in curated_sample[:5]:
    print(item)


In [17]:
train_items, validation_items, test_items = split_items(
    curated_sample,
    train_ratio=0.98,
    validation_ratio=0.01,
    seed=42,
)

In [18]:
print(len(train_items))

9800


In [ ]:
username = "mainanorbert"
repo_id = f"{username}/items_raw_curated"
push_curated_splits(repo_id, train_items, validation_items, test_items)

In [ ]:
# Configure and run curation pipeline
username = "mainanorbert"
sample_size = 300000

authenticate_huggingface()
dataset = load_raw_appliances_dataset()
items = parse_valid_items(dataset)
curated_sample = build_weighted_sample(items, sample_size=sample_size, seed=42)

train_items, validation_items, test_items = split_items(
    curated_sample,
    train_ratio=0.98,
    validation_ratio=0.01,
    seed=42,
)

repo_id = f"{username}/items_raw_curated"
push_curated_splits(repo_id, train_items, validation_items, test_items)